# Introduction to SEM-EDX data processing using HyperSpy and Jupyter Notebooks

HyperSpy can be easily used on data with more dimension than the classical 3D EDS mapping. EDS/SEM can be extended to three-dimensional microanalysis with a "slice-and-view" approach. With such 4D EDS/SEM mapping, a 3D stack of X-ray line intensity maps can be extracted and plotted, as shown in this notebook with data acquired from a Ni-based superalloy.

## Author

* 13/04/2015 Pierre Burdet - Developed for HyperSpy workshop at University of Cambridge

## Changes

* 04/06/2025 Magnus Nord. Adapted for SCANDEM 2025, added part of making figures using matplotlib.
* 06/05/2022 Magnus Nord. Adapted to show and explain basic Python and HyperSpy concepts. Originally for Nordic Nanolab User Meeting (NNUM) 2022. For the original notebook, with more advanced processing, see the [HyperSpy demos](https://github.com/hyperspy/exspy-demos/blob/main/EDS/SEM_EDS_4D_visualisation.ipynb)
* 28/02/2017 Francisco de la Peña. Update the syntax for HyperSpy 1.2.
* 03/08/2016 Francisco de la Peña. Update the syntax for HyperSpy 1.1 and minor improvements to layout.
* 27/08/2016 Pierre Burdet. Update for workshop at EMC Lyon

## Requirements

* Used and tested with HyperSpy 1.7.0, might work with other versions

## The data

The sample and the data used in this tutorial are described in 
P. Burdet, et al., Ultramicroscopy, 148, p. 158-167 (2015) (see the [abstract](http://www.sciencedirect.com/science/article/pii/S0304399114002022)).

Ni-based superalloy sample is investigated with 4D EDS/SEM experiment using a FIB/SEM (Helios FEI). The data are exported from INCA software (Oxford instrument). The ".msa" file contains a single pixel spectrum and the metadata to calibrate it. The ".rpl" is a header to read the ".raw". Each ".raw" file contains an EDS/SEM spectral image, acquired at different milled thicknesses (depth or z axis). The ".tiff" files contains the corresponding SE images.

## Scaling and calibrating the raw data

First we need to set a plotting library

In [ ]:
%matplotlib qt

Then we need to import the HyperSpy library

In [ ]:
import hyperspy.api as hs

### Loading

Now, we're going to look at a slice-and-view dataset. To do this, we use HyperSpy's load method. `load` returns a signal, which must be assigned to a variable.

In [ ]:
s = hs.load("datasets/Ni_superalloy_010.rpl")

Now lets have a look at this dataset, lets see what `s` is

In [ ]:
s

Here we see that we have a Signal2D `(1024|256, 224)`. The numbers to the left of the `|` is the navigation dimension, while the numbers to the right of the `|` is the signal dimensions. The easiest way to show this, is by visualizing the data through the `plot` function in the signal `s`.

**Note:** the plotting window might be hidden behind the notebook window.

In [ ]:
s.plot()

Here we get two separate windows: one showing the navigation dimension, and one showing the signal dimensions. We see that the navigation dimension has an Energy Dispersive X-ray Spectrum, but the signal dimension is rather boring at the moment.

The navigation dimension has a *navigator*, show here with the red line. It can be moved either by using the **left or right keys on your keyboard**, or we can drag it using the mouse. This enables us to see where the different EDS peaks originate from. We'll have a look at this later.

**Note:** this navigator line is difficult to see, it is all the way to the left in the spectrum plot.

#### Loading the other EDS slices

Lets load the rest of the slices.

In [ ]:
s11 = hs.load("datasets/Ni_superalloy_011.rpl")

In [ ]:
s12 = hs.load("datasets/Ni_superalloy_012.rpl")

And lets have a look at the shape and size of these

In [ ]:
s11

In [ ]:
s12

Both have exactly the same shape as the one we first looked at.

#### Stacking the data

Since these 3 datasets are from a slice-and-view experiment, with exactly the same size, we can combine them all into the same signal. To do this, we use the `stack` function.

In [ ]:
s = hs.stack([s, s11, s12])

In [ ]:
s

Now we have a 4th dimension, in addition the 3 ones we saw earlier. Lets see how this looks like.

In [ ]:
s.plot()

To process the data, we need to view them as spectrum per probe position, rather than probe position per energy channel.

To do this, we need to change which dimensions are navigation dimensions, and which one is the signal dimension. This is done via the `transpose` function.

In [ ]:
st = s.transpose(signal_axes=(0, ))

In [ ]:
st

The ability to switch around which dimension is navigation and signal is really powerful!

Now, the x-ray energy dimension is the only signal dimension. Using plot, we see how it looks like. This gives us an extra window, allowing us to navigate across the 3 slices. Also, notice that we do not have a calibrated dataset: the EDS axis just shows energy channels, and the probe dimensions show pixels. We'll address this soon.

Tips: use the + key to increase the size of the navigator, note that this does not mean it averages over several spectra. It is just a convenience to make it easier to select. Another tip is pressing the `E` key to get a second navigator, which makes it easier to compare different positions.

In [ ]:
st.plot()

### Setting the signal type

The signal `st` we've been looking at, is a `Signal1D`. This is a class object, which contains many different functions, like `plot` which we saw earlier.

This is a rather generic signal type, but we have many others with more specialized functionality. We can see them all via:

In [ ]:
hs.print_known_signal_types()

There are quite a few, especially for electron microscopy data. The one we want is `EDS_SEM`. To change it, we use `set_signal_type`

In [ ]:
st.set_signal_type('EDS_SEM')

In [ ]:
st

So now we have an `EDSSEMSpectrum`, instead of `Signal1D`. This gives us access to some additional EDS-specific functions.

We can see all the functions which are contained in the signals, by writing the signal object, a `.` then pressing the TAB key (typically above the `Caps Lock` key).

If you for example want to see what plotting functions are available, you can write `st.plot` then press `TAB`.

In [ ]:
st.

### Calibrating the signal

Now, lets calibrate our data.

To get this information, we use a little trick by using a "one pixel" spectrum, in the `.msa` format. This gives us both the energy axis scaling and microscope parameters.

In [ ]:
s1p = hs.load('datasets/Ni_superalloy_1pix.msa', signal_type='EDS_SEM')

In [ ]:
st.get_calibration_from(s1p)

Here we used a function to copy the energy calibration and microscope parameters. The microscope parameters, we can see in the `metadata` attribute in `s`

In [ ]:
st.metadata

Lastly, we set the probe step size calibration. These values we get directly from the microscope software. `x`: units `um`, scale = `0.05`. `y`: units `um`, scale = `0.05`. `z`: units `um`, scale = `0.1`. Axis 4: name, `Energy`

In [ ]:
st.axes_manager.gui()

Plotting the data, we can now see that both the probe and energy axis are calibrated.

In [ ]:
st.plot()

### Saving the data

Now that we've both loaded all the data and calibrated it, we can save it in a better format: HyperSpy's version of HDF5 `.hspy`

Saving is easy, we just use the `.save` method is the signal.

In [ ]:
st.save("Ni_superalloy.hspy")

We load this file using the same `hs.load` function as earlier.

In [ ]:
s2 = hs.load("Ni_superalloy.hspy")

It keeps both the correct data size, and the calibrations

In [ ]:
s2.axes_manager

## Visualisation

### X-ray lines

With the data calibrated, we can now get some elemental maps. First, we need to tell HyperSpy which elements we think we have in the sample.

In [ ]:
s2.add_elements(('Al', 'C', 'Co', 'Cr', 'Mo', 'Ni', 'Ta', 'Ti'))

We can see in the `metadata` which elements have been added

In [ ]:
s2.metadata

We can visualize this on the dataset, by using the `xray_lines` parameter

In [ ]:
s2.plot(xray_lines=True)

However, a better way of doing this is by plotting this on a sum of the data. There are several ways of doing this, but I'll show you a trick, which I call command chaining.

* First, we'll sum the data across all the navigation positions.
* This gives us a signal with no navigation dimensions, and 1 signal dimension
* Then, we'll place a `plot(xray_lines=True)` directly after the `sum()`

In [ ]:
s2.sum().plot(xray_lines=True)

### Making elemental map

Now, lets make some better elemental maps, since the ones we plotted earlier only used a single energy channel. First, lets do this for the first slice.

There are several ways we could do this, but lets use a concept called slicing. Lets see exactly what we want to do.

In [ ]:
s2

#### Interlude: how to slice

We want to get the first index in the last navigation dimension. To do this, we'll use the `inav` function, which we can use to crop the data in the navigation dimensions.

* Need to use square brackets `[]`
* `:` means we select all the positions in that dimension
* Python is `zero indexed`, meaning the first index is 0.

In [ ]:
s2.inav[:, :, 0]

While this gives us what we need, I think it is useful to show you the other things you can do using slicing, since it is such a powerful tool for working with your data.

For example, we can crop the data across multiple dimensions at the same time, to get several indices.

In [ ]:
s2.inav[0:120, 90:160, 0].plot()

There is a separate function for slicing in the signal dimension: `isig`. It works just the same as `inav`.

In [ ]:
s2.isig[512:1024]

It is also possible to combine these, by chaining them. For example, we can extract a single spectrum from a single probe position.

In [ ]:
s2.inav[100, 150, 2].isig[512:1024].plot()

Note that we up until now have done this slicing in uncalibrated units, like energy channel and pixels, NOT keV or micrometers. This is given by the numbers being either integers, or floats.

If we plot it, we can see that the energy axis is from 5.2 to 7.9 keV.

In [ ]:
s2.isig[5.2:7.9]

This syntax is very similar to other python libraries, like NumPy, so even if this syntax might look a bit complicated, I think it is very useful to learn.

#### Getting the elemental maps

But lets get back to making some elemental map!

In [ ]:
lines_intensity = s2.get_lines_intensity()

This gives us a list of signals, where each signal shows a different element.

In [ ]:
lines_intensity

We access the different signals in the list using the `[]` syntax. Remember that we use zero indexing, ergo, if we want the `Al_Ka` line, we need to use `[0]`.

Note that we now have signals with 3 navigation dimensions, and no signal dimensions. So we should `transpose` these to get the **x** and **y** axes as signal dimensions, and **z** as the navigation dimension.

In [ ]:
s_cr = lines_intensity[3].transpose(signal_axes=('x', 'y'))

In [ ]:
s_cr.plot()

In [ ]:
s_ti = lines_intensity[-1].transpose(signal_axes=('x', 'y'))

In [ ]:
s_ti.plot()

# Making nice figures

### Simple figure

In [ ]:
s_cr0 = s_cr.inav[0]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots()

In [ ]:
ax.imshow(s_cr.inav[0])

In [ ]:
fig.savefig("eds_cr.jpg")

Looking at the figure it shows the data, but needs some improvements.

- Scalebar
- Low resolution
- Lots of whitespace
- Remove numbers on axis

### Improving the figure

In [ ]:
fig, ax = plt.subplots()

In [ ]:
ax.imshow(s_cr0, extent=s_cr0.axes_manager.signal_extent, cmap='inferno')

In [ ]:
ax.xaxis.set_ticks([])
ax.yaxis.set_ticks([])

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

In [ ]:
scalebar = AnchoredSizeBar(ax.transData, size=3, label="3 um", loc='lower right', frameon=False, size_vertical=0.2, color='white', borderpad=0.2)

In [ ]:
ax.add_artist(scalebar)

In [ ]:
fig.tight_layout()

In [ ]:
fig.savefig("better_figure.jpg", dpi=300)

### Adding more elements

In [ ]:
fig, axarr = plt.subplots(1, 2)

In [ ]:
ax_cr = axarr[0]
ax_ti = axarr[1]

In [ ]:
ax_cr.imshow(s_cr0, extent=s_cr0.axes_manager.signal_extent, cmap='inferno')

In [ ]:
ax_ti.imshow(s_ti.inav[0], cmap='viridis')

In [ ]:
scalebar = AnchoredSizeBar(ax_cr.transData, size=3, label="3 um", loc='lower right', frameon=False, size_vertical=0.2, color='white', borderpad=0.2)

In [ ]:
ax_cr.add_artist(scalebar)

In [ ]:
for ax in axarr:
    ax.xaxis.set_ticks([])
    ax.yaxis.set_ticks([])

In [ ]:
ax_cr.annotate(text="Cr", xy=(0.05, 0.9), textcoords='axes fraction', color='white', fontsize=20)

In [ ]:
ax_ti.annotate(text="Ti", xy=(0.05, 0.9), textcoords='axes fraction', color='white', fontsize=20)

In [ ]:
fig.tight_layout()

In [ ]:
fig.savefig("multiple_elements.jpg", dpi=300)

## Plotting with a different navigator

Another signal, such as secondary electron (SE) images can be used as a navigator for the spectrum data.

In [ ]:
im = hs.load("datasets/Ni_superalloy_010.tif")

This signal works pretty much the same as the other signals we've looked at, but it has 2 signal dimensions, and 0 navigation dimensions.

In [ ]:
im.plot()

In [ ]:
im

However, to use it as navigator for the spectrum data, we need to first make it same size as the navigation dimensions in the spectrum data.

Here, we see that the `x` and `y` sizes are 4 times as large as the probe positions in the spectral data. Thus, we need to combine 16 pixels (4 x 4) in the SE images.

In [ ]:
im_rebin = im.rebin(scale=(4, 4))

In [ ]:
im_rebin

Now we can use this as a navigator for our EDS signal.

In [ ]:
s20 = s2.inav[:, :, 0]

In [ ]:
s20.plot(navigator=im_rebin)

We can also use the elemental maps we made as navigators

In [ ]:
s2.plot(navigator=s_cr)

# Further work

This notebook covered the basics of using HyperSpy for analysing EDS data. For more advanced functionality, see [HyperSpy's EDS documentation](https://hyperspy.org/exspy/user_guide/eds.html). You can also have a look at the [original version of this notebook](https://github.com/hyperspy/exspy-demos/blob/main/EDS/SEM_EDS_4D_visualisation.ipynb), which covers some more advanced processing.

The full version of the [HyperSpy documentation](https://hyperspy.org/hyperspy-doc/current/index.html), contain information about:

* The [data visualization](https://hyperspy.org/hyperspy-doc/current/user_guide/visualisation.html) functions we used
* If you have really big datasets, you can use the [lazy functionality](https://hyperspy.org/hyperspy-doc/current/user_guide/big_data.html)
* [Principle component analysis](https://hyperspy.org/hyperspy-doc/current/user_guide/mva/index.html), which we used for denoising
* Aligning the different FIB slices by using the `align2D` method. Here one can use the secondary electron images to estimate the shifts, then apply them to the EDS data

For matplotlib, there is a a very nice gallery showing the different types of figures you can make: https://matplotlib.org/stable/gallery/